In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import ArtistAnimation
from IPython.display import HTML

Initial Condition

Left State:

    rhoL = 1.0 

    velL = 0.0

    preL = 1.0

Right State:

    rhoR = 0.125

    velR = 0.0

    preR = 0.1


## HLL

In [17]:
def prepareanimation():
    global figA, artistA
    #set up for animation
    figA = plt.figure(figsize=(8,6))
    artistA = []

    plt.xlim(-0.5*1,0.5*1)
    plt.xlabel(r'$x$')

In [18]:
def addanimation(cel_center, rho, u, p, legend):

    global artistA
    
    im1 = plt.plot(cel_center, rho, color='blue',label=r"$\rho$",)
    im2 = plt.plot(cel_center, u, color='red',label=r"$v$",)
    im3 = plt.plot(cel_center, p, color='green',label=r"$P$")

    plt.xlabel(r'$x$')
    plt.xlim(min(cel_center), max(cel_center))
    plt.ylim(-0.1, 1.2)
    
    if legend == 1:
        plt.legend()

    figA.tight_layout()
    artistA.append(im1 + im2 + im3)

In [19]:
def godunov(ist, ien, dt, dx, rho, mom,  ene, frho, fmom, fe):
    
    for i in range(ist, ien-1):
        rho[i] = rho[i] - dt/dx*( frho[i] - frho[i-1] )
        mom[i] = mom[i] - dt/dx*( fmom[i] - fmom[i-1] )
        ene[i] = ene[i] - dt/dx*( fe[i] - fe[i-1] )

    return rho,mom,ene     

In [20]:
def newtonraphson(uL, uR, pR, pL, cL, cR):
    global gamma
    ## vL(p)=vR(p)
    #f(p) = vL(p) - vR(p)
    gamma = 1.4

    p_i = ( cR*pL + cL*pR - cR*cL*(uR - uL) )/(cR + cL)
    
    if p_i<0:p_i = 1e-8
    
    delta = 10
    dp_Ldv=0
    dp_Rdv=0
    vL = 0
    vR = 0
    
    while abs(delta) > 1e-4:
        mass_flux_L = 0.5*cL**2*((gamma+1)*p_i/pL + (gamma -1))/gamma
        dp_Ldv = 2*mass_flux_L*np.sqrt(mass_flux_L)/(mass_flux_L + cL**2)
        
        mass_flux_R = 0.5*cR**2*((gamma+1)*p_i/pR + (gamma -1))/gamma
        dp_Rdv = 2*mass_flux_R*np.sqrt(mass_flux_R)/(mass_flux_R + cR**2)
        
        vL = uL - (p_i - pL)/np.sqrt(mass_flux_L)
        vR = uR + (p_i - pR)/np.sqrt(mass_flux_R) 

        delta = dp_Ldv* dp_Rdv*(vR - vL)/ ( (dp_Ldv + dp_Rdv)*p_i)
        p_i = p_i*(1.0 - delta)

    return p_i, (dp_Ldv*vL+ dp_Rdv*vR)/(dp_Ldv + dp_Rdv)

In [21]:
def hll(rhoL, uL, pL, rhoR, uR, pR):

    global gamma

    rhoL = rhoL
    uL = uL
    pL = pL 

    rhoR = rhoR
    uR = uR
    pR = pR

    gamma = 1.4
   
    s = np.zeros(3)

    cL = np.sqrt(gamma*pL*rhoL)

    cR = np.sqrt(gamma*pR*rhoR)

    p_star, v_star = newtonraphson(uL, uR, pR, pL, cL, cR)
    

    aL = np.sqrt(gamma * pL / rhoL)
    aR = np.sqrt(gamma * pR / rhoR)
    
    if p_star >= pL: #shock
        rho_star_l = rhoL*(p_star/pL + (gamma-1)/(gamma+1))/( (gamma-1)/(gamma+1)*p_star/pL + 1.0)
        s[0] = uL - aL*np.sqrt( ( (gamma + 1)*p_star/pL + (gamma+1) )/(2.0*gamma) )
        
    else: #rarefaction
        rho_star_l = rhoL * (p_star/pL)**(1/gamma)
        s[0] = 0.5*( uL - aL + v_star - np.sqrt(gamma*p_star/rho_star_l) )
        

    if p_star >= pR: #shock
        rho_star_r = rhoR*(p_star/pR + (gamma-1)/(gamma+1))/( (gamma-1)/(gamma+1)*p_star/pR + 1.0)
        s[2] = uR + aR*np.sqrt( ( (gamma + 1)*p_star/pR + (gamma+1) )/(2.0*gamma) )
        
    else: #rarefaction
        rho_star_r = rhoR * (p_star/pR)**(1/gamma)
        s[2] = 0.5*( uR + aR + v_star + np.sqrt(gamma*p_star/rho_star_r) )
        


    #s[0] = min(uL - np.sqrt(gamma*pL/rhoL), uR - np.sqrt(gamma*pR/rhoR))
    #s[2] = max(uL + np.sqrt(gamma*pL/rhoL), uR + np.sqrt(gamma*pR/rhoR))

    
    s[1] = v_star


    if s[0] > 0: #wL
        frho =  rhoL * uL
        fmom= rhoL * uL**2 + pL
        fe = (gamma*pL/(gamma-1) + 0.5*rhoL*uL**2) * uL

        
    elif s[0]<=0 and s[2] >= 0: #w*
        frho =  (s[2]*rhoL * uL - s[0]*rhoR * uR + s[2]*s[0]*(rhoR - rhoL))/(s[2]-s[0])
        fmom= (s[2]*(rhoL * uL**2 + pL) - s[0]*(rhoR * uR**2 + pR) + s[2]*s[0]*(rhoR*uR - rhoL*uL))/(s[2]-s[0])
        fe = (s[2]*(gamma*pL/(gamma-1) + 0.5*rhoL*uL**2) * uL - s[0]*(gamma*pR/(gamma-1) + 0.5*rhoR*uR**2) * uR + s[2]*s[0]*(pR/(gamma-1)+ 0.5*rhoR*uR**2 - pL/(gamma-1)+ 0.5*rhoL*uL**2))/(s[2]-s[0])
           
    else:  #wR
        frho =  rhoR * uR
        fmom= rhoR * uR**2 + pR
        fe = (gamma*pR/(gamma-1) + 0.5*rhoR*uR**2) * uR

    return frho, fmom, fe

In [23]:
def shocktube1d(rhoL,velL,preL,rhoR,velR,preR,L,ngrids,cfl,tout):
    
    global ist, ien, dx, ntot, gamma

    #initial conditions
    rhoL = rhoL 
    uL = velL
    pL = preL

    rhoR = rhoR
    uR = velR
    pR = preR

    dx = L / ngrids
    cfl = 0.8
    gamma = 1.4
     
    ist = 2
    nghost = ist
    ien = ngrids + nghost
    ntot = ngrids + 2*nghost
    cfl = 0.5

    rho = np.zeros(ntot)
    u = np.zeros(ntot)
    p = np.zeros(ntot)

    #fluxes
    global frho, frhou, fe
    frho = np.zeros(ntot)
    fmom = np.zeros(ntot)
    fe = np.zeros(ntot)

    mom = np.zeros(ntot)
    ene = np.zeros(ntot)

    time = 0
    dt_plot = 0.01


    prepareanimation()


    cel_center = np.zeros(ntot)
    cel_boundary = np.zeros(ntot+1)

    for i in range(ntot +1):
        cel_boundary[i] = -0.5*L + (i-ist)*dx

    for i in range(ntot):
        cel_center[i] = 0.5*(cel_boundary[i] + cel_boundary[i+1])

    #profile at t = 0
    x0 = ngrids//2.0 # center of the shock tube

    for i in range(ntot):
        
        if i <= x0:
            rho[i] = rhoL
            u[i] = uL
            p[i] = pL

        else:
            rho[i] = rhoR
            u[i] = uR
            p[i] = pR

        mom[i] = rho[i]*u[i]
        ene[i] = p[i]/(gamma-1) + 0.5*rho[i]*u[i]**2    
        """
        rho[i] = (rhoR - rhoL)*0.5*( np.tanh(cel_center[i]/(1.0*dx)) + 1.0) + rhoL
        u[i] = (velR - velL)*0.5*( np.tanh(cel_center[i]/(1.0*dx)) + 1.0) + velL
        p[i] = (preR - preL)*0.5*( np.tanh(cel_center[i]/(1.0*dx)) + 1.0) + preL
        """
        

    
    addanimation(cel_center, rho, u, p,1)


    rhoL = np.zeros(ntot)   
    rhoR = np.zeros(ntot)  
    uL = np.zeros(ntot) 
    uR = np.zeros(ntot)   
    pL = np.zeros(ntot)  
    pR = np.zeros(ntot)   


    dt = cfl*np.min(dx/(np.sqrt(gamma*p[ist:ien]/rho[ist:ien ]) + abs(u[ist:ien])))
    

    time_plot = 0

    while time < tout:
        
            # calculate w_{i+1/2,L}
        for i in range(ist-1,ien):
            rhoL[i] = rho[i]
            uL[i] = u[i]
            pL[i] = p[i]

        # calculate u_{i+1/2,R}
        for i in range(ist-1,ien):
            rhoR[i] = rho[i+1]
            uR[i] = u[i+1]
            pR[i] = p[i+1]

        for i in range(ist-1, ien):
            frho[i], fmom[i], fe[i] = hll(rhoL[i], uL[i], pL[i], rhoR[i], uR[i], pR[i])
        
        rho, mom, ene = godunov(ist, ien, dt, dx, rho, mom,  ene, frho, fmom, fe)

        for i in range(ist, ien):
            u[i] = mom[i]/rho[i]
            p[i] = (gamma-1)*(ene[i] - 0.5*rho[i]*u[i]**2)  #(ene[i] - 0.5*mom[i]*u[i])/rho[i]
        
        time = time + dt
        time_plot = time_plot + dt
        
        if time_plot > dt_plot:
            addanimation(cel_center, rho, u, p,0)
            time_plot = 0.0

        dt = cfl*np.min(dx/(np.sqrt(gamma*p[ist:ien]/rho[ist:ien ]) + abs(u[ist:ien])))
        
        if  time + dt > tout:
            dt = tout - time
        
    addanimation(cel_center, rho, u, p,0)


if __name__ == "__main__":
    shocktube1d(rhoL = 1.0 ,velL= 0.0, preL=1.0 ,rhoR=0.125 ,velR=0.0 ,preR=0.1,L=1.0,ngrids=64,cfl=0.2,tout=0.20)

anim = ArtistAnimation(figA, artistA, interval=100) 
plt.close(figA) 
HTML(anim.to_jshtml())
 

## HLLC

In [2]:
def prepareanimation():
    global figA, artistA
    #set up for animation
    figA = plt.figure(figsize=(8,6))
    artistA = []

    #plt.xlim(-0.5*1,0.5*1)
    plt.xlabel(r'$x$')

In [13]:
def addanimation(cel_center, rho, u, p, legend):

    global artistA
    
    im1 = plt.plot(cel_center, rho, color='red',label=r"$\rho$",)
    im2 = plt.plot(cel_center, u, color='green',label=r"$v$",)
    im3 = plt.plot(cel_center, p, color='blue',label=r"$P$")

    plt.xlabel(r'$x$')
    plt.xlim(min(cel_center), max(cel_center))
    #plt.ylim(-0.1, 1.2)
    
    if legend == 1:
        plt.legend()

    figA.tight_layout()
    
    artistA.append(im1 + im2 + im3)

In [5]:
def godunov(ist, ien, dt, dx, rho, mom,  ene, frho, fmom, fe):
    #print(len(frho))
    
    for i in range(ist, ien):

        rho[i] = rho[i] - dt/dx*( frho[i] - frho[i-1] )
        mom[i] = mom[i] - dt/dx*( fmom[i] - fmom[i-1] )
        ene[i] = ene[i] - dt/dx*( fe[i] - fe[i-1] ) 

    return rho,mom,ene 

In [6]:
def newtonraphson(uL, uR, pR, pL, cL, cR):
    global gamma
    ## vL(p)=vR(p)
    #f(p) = vL(p) - vR(p)
    gamma = 1.4

    p_i = ( cR*pL + cL*pR - cR*cL*(uR - uL) )/(cR + cL)
    
    if p_i<0:p_i = 1e-8
    
    delta = 10
    dp_Ldv=0
    dp_Rdv=0
    vL = 0
    vR = 0
    
    while abs(delta) > 1e-4:
        mass_flux_L = 0.5*cL**2*((gamma+1)*p_i/pL + (gamma -1))/gamma
        dp_Ldv = 2*mass_flux_L*np.sqrt(mass_flux_L)/(mass_flux_L + cL**2)
        
        mass_flux_R = 0.5*cR**2*((gamma+1)*p_i/pR + (gamma -1))/gamma
        dp_Rdv = 2*mass_flux_R*np.sqrt(mass_flux_R)/(mass_flux_R + cR**2)
        
        vL = uL - (p_i - pL)/np.sqrt(mass_flux_L)
        vR = uR + (p_i - pR)/np.sqrt(mass_flux_R) 

        delta = dp_Ldv* dp_Rdv*(vR - vL)/ ( (dp_Ldv + dp_Rdv)*p_i)
        p_i = p_i*(1.0 - delta)

    return p_i, (dp_Ldv*vL+ dp_Rdv*vR)/(dp_Ldv + dp_Rdv)

In [7]:
def hllc(rhoL, uL, pL, rhoR, uR, pR):

    global gamma

    rhoL = rhoL
    uL = uL
    pL = pL 

    rhoR = rhoR
    uR = uR
    pR = pR

    gamma = 1.4
   
    s = np.zeros(3)

    cL = np.sqrt(gamma*pL*rhoL)

    cR = np.sqrt(gamma*pR*rhoR)
     
    p_star, v_star = newtonraphson(uL, uR, pR, pL, cL, cR)
    

    aL = np.sqrt(gamma * pL / rhoL)
    aR = np.sqrt(gamma * pR / rhoR)
    
    if p_star >= pL: #shock
        rho_star_l = rhoL*(p_star/pL + (gamma-1)/(gamma+1))/( (gamma-1)/(gamma+1)*p_star/pL + 1.0)
        s[0] = uL - aL*np.sqrt( ( (gamma + 1)*p_star/pL + (gamma+1) )/(2.0*gamma) )
        
    else: #rarefaction
        rho_star_l = rhoL * (p_star/pL)**(1/gamma)
        s[0] = 0.5*( uL - aL + v_star - np.sqrt(gamma*p_star/rho_star_l) )
        

    if p_star >= pR: #shock
        rho_star_r = rhoR*(p_star/pR + (gamma-1)/(gamma+1))/( (gamma-1)/(gamma+1)*p_star/pR + 1.0)
        s[2] = uR + aR*np.sqrt( ( (gamma + 1)*p_star/pR + (gamma+1) )/(2.0*gamma) )
        
    else: #rarefaction
        rho_star_r = rhoR * (p_star/pR)**(1/gamma)
        s[2] = 0.5*( uR + aR + v_star + np.sqrt(gamma*p_star/rho_star_r) )
        


    #s[0] = min(uL - np.sqrt(gamma*pL/rhoL), uR - np.sqrt(gamma*pR/rhoR))
    #s[2] = max(uL + np.sqrt(gamma*pL/rhoL), uR + np.sqrt(gamma*pR/rhoR))

    
    s[1] = v_star


    if s[0] >= 0: #wL
        frho =  rhoL * uL
        fmom= rhoL * uL**2 + pL
        fe = (gamma*pL/(gamma-1) + 0.5*rhoL*uL**2) * uL

    elif s[1] >= 0: #wL*
        frho =  rho_star_l * v_star
        fmom= rho_star_l * v_star**2 + p_star
        fe = (gamma*p_star/(gamma-1) + 0.5*rho_star_l*v_star**2) * v_star

    elif s[2] >= 0: #wR*
        frho =  rho_star_r * v_star
        fmom= rho_star_r * v_star**2 + p_star
        fe = (gamma*p_star/(gamma-1) + 0.5*rho_star_r*v_star**2) * v_star

    else:  #wR
        frho =  rhoR * uR
        fmom= rhoR * uR**2 + pR
        fe = (gamma*pR/(gamma-1) + 0.5*rhoR*uR**2) * uR

    return frho, fmom, fe

In [24]:
def shocktube1d(rhoL,velL,preL,rhoR,velR,preR,L,ngrids,cfl,tout):
    
    global ist, ien, dx, ntot, gamma

    #initial conditions
    rhoL = rhoL 
    uL = velL
    pL = preL

    rhoR = rhoR
    uR = velR
    pR = preR

    dx = L / ngrids
    gamma = 1.4
     
    ist = 2
    nghost = ist
    ien = ngrids + nghost
    ntot = ngrids + 2*nghost

    rho = np.zeros(ntot)
    u = np.zeros(ntot)
    p = np.zeros(ntot)

    #fluxes
    global frho, frhou, fe
    frho = np.zeros(ntot)
    fmom = np.zeros(ntot)
    fe = np.zeros(ntot)

    mom = np.zeros(ntot)
    ene = np.zeros(ntot)

    time = 0
   
    prepareanimation()


    cel_center = np.zeros(ntot)
    cel_boundary = np.zeros(ntot+1)

    for i in range(ntot +1):
        cel_boundary[i] = -0.5*L + (i-ist)*dx #(i - ist) * dx

    for i in range(ntot):
        cel_center[i] = 0.5*(cel_boundary[i] + cel_boundary[i+1])

    #profile at t = 0
    x0 = ngrids//2.0 # center of the shock tube

    for i in range(ntot):
        
        if i <= x0:
            rho[i] = rhoL
            u[i] = uL
            p[i] = pL

        else:
            rho[i] = rhoR
            u[i] = uR
            p[i] = pR
        """
        rho[i] = (rhoR - rhoL)*0.5*( np.tanh(cel_center[i]/(1.0*dx)) + 1.0) + rhoL
        u[i] = (velR - velL)*0.5*( np.tanh(cel_center[i]/(1.0*dx)) + 1.0) + velL
        p[i] = (preR - preL)*0.5*( np.tanh(cel_center[i]/(1.0*dx)) + 1.0) + preL
        """
        mom[i] = rho[i]*u[i]
        ene[i] = p[i]/(gamma-1) + 0.5*rho[i]*u[i]**2 
           
        
       
    addanimation(cel_center, rho, u, p,1)


    rhoL = np.zeros(ntot)   
    rhoR = np.zeros(ntot)  
    uL = np.zeros(ntot) 
    uR = np.zeros(ntot)   
    pL = np.zeros(ntot)  
    pR = np.zeros(ntot)   


    dt = cfl*np.min(dx/(np.sqrt(gamma*p[ist:ien]/rho[ist:ien ]) + abs(u[ist:ien])))
    

    time_plot = 0
    dt_plot = 0.01

    while time < tout:
        
        # calculate w_{i+1/2,L}
        for i in range(ist-1,ien):
            rhoL[i] = rho[i]
            uL[i] = u[i]
            pL[i] = p[i]

        # calculate u_{i+1/2,R}
        for i in range(ist-1,ien):
            rhoR[i] = rho[i+1]
            uR[i] = u[i+1]
            pR[i] = p[i+1]

        for i in range(ist-1, ien):
            frho[i], fmom[i], fe[i] = hllc(rhoL[i], uL[i], pL[i], rhoR[i], uR[i], pR[i])
        
        rho, mom, ene = godunov(ist, ien, dt, dx, rho, mom,  ene, frho, fmom, fe)

        for i in range(ist, ien):
            u[i] = mom[i]/rho[i]
            p[i] = (gamma-1)*(ene[i] - 0.5*rho[i]*u[i]**2)  #(ene[i] - 0.5*mom[i]*u[i])/rho[i]
        
        time = time + dt
        time_plot = time_plot + dt
        
        if time_plot > dt_plot:
            addanimation(cel_center, rho, u, p,0)
            time_plot = 0.0

        dt = cfl*np.min(dx/(np.sqrt(gamma*p[ist:ien]/rho[ist:ien ]) + abs(u[ist:ien])))
        
        if  time + dt > tout:
            dt = tout - time
        
    addanimation(cel_center, rho, u, p,0)



shocktube1d(rhoL = 1.0 ,velL= 0.0, preL=1.0 ,rhoR=0.125 ,velR=0.0 ,preR=0.1,L=1,ngrids=64,cfl=0.2,tout=0.2)

anim = ArtistAnimation(figA, artistA, interval=100) 
plt.close(figA) 
HTML(anim.to_jshtml())
 

Initial Condition

Left State:

    rhoL = 1.0 

    velL = 0.0

    preL = 0.1

Right State:

    rhoR = 0.125

    velR = 0.0

    preR = 1.0


In [10]:
shocktube1d(rhoL = 1.0 ,velL= 0.0, preL=0.1 ,rhoR=0.125 ,velR=0.0 ,preR=1.0,L=1,ngrids=64,cfl=0.2,tout=0.2)

anim = ArtistAnimation(figA, artistA, interval=100) 
plt.close(figA) 
HTML(anim.to_jshtml())

Initial Condition

Left State:

    rhoL = 1.0 

    velL = 0.0

    preL = 1.0

Right State:

    rhoR = 0.125

    velR = 0.0

    preR = 1.0


In [11]:
shocktube1d(rhoL = 1.0 ,velL= 0.0, preL=1.0 ,rhoR=0.125 ,velR=0.0 ,preR=1.0, L=1, ngrids=64, cfl=0.2, tout=0.2)

anim = ArtistAnimation(figA, artistA, interval=100) 
plt.close(figA) 
HTML(anim.to_jshtml())

Initial Condition

Left State:

    rhoL = 1.0 

    velL = 0.0

    preL = 1.2

Right State:

    rhoR = 0.125

    velR = 0.0

    preR = 1.0


In [15]:
shocktube1d(rhoL = 1.0 ,velL= 0.0, preL=1.5 ,rhoR=0.125 ,velR=0.0 ,preR=1.0, L=1, ngrids=64, cfl=0.2, tout=0.2)

anim = ArtistAnimation(figA, artistA, interval=100) 
plt.close(figA) 
HTML(anim.to_jshtml())

## Exact Riemann solver

In [3]:
def prepareanimation():
    global figA, artistA
    #set up for animation
    figA = plt.figure(figsize=(8,6))
    artistA = []

    #plt.xlim(-0.5*1,0.5*1)
    plt.xlabel(r'$x$')

In [4]:
def addanimation(cel_center, rho, u, p, legend):

    global artistA
    
    im1 = plt.plot(cel_center, rho, color='red',label=r"$\rho$",)
    im2 = plt.plot(cel_center, u, color='green',label=r"$v$",)
    im3 = plt.plot(cel_center, p, color='blue',label=r"$P$")

    plt.xlabel(r'$x$')
    plt.xlim(min(cel_center), max(cel_center))
    #plt.ylim(-0.1, 1.2)
    
    if legend == 1:
        plt.legend()

    figA.tight_layout()
    
    artistA.append(im1 + im2 + im3)

In [ ]:
def exactsolution(L, tout, rhoL, velL, preL, rhoR, velR, preR):
    global gamma,   gamma1, gamma2, gamma3, gamma4, gamma5
    #constants
    gamma = 1.4
    gamma1 = gamma + 1
    gamma2 = gamma -1
    gamma3 = gamma2/gamma1
    gamma4 = gamma2/(2*gamma)
    gamma5 = gamma1/(2*gamma)


    #box
    L = 1.0
    ngrids = 64
    ghost = 2
    ntot = ngrids + 2*ghost
    dx = L/ntot
    ist = ghost
    ien = ngrids + ghost


    tout = tout
    cfl = 0.5

    celboundary = np.zeros(ntot + 1)
    celcenter = np.zeros(ntot)


    for i in range(ntot+1):
        celboundary[i] = -0.5*L + (i - ghost)*dx

    for i in range(ntot + 1):
        celcenter = (celboundary[i] - celboundary[i+1])/2

    rho = np.zeros(ntot)
    u = np.zeros(ntot)
    p = np.zeros(ntot)
    aL = np.sqrt(gamma*pL/rhoL)
    aR = np.sqrt(gamma*pR/rhoR)

    #initial conditions
    for i in range(ntot):
        rho[i] = (rhoR - rhoL)*0.5*( np.tanh(celcenter[i]/(1.0*dx)) + 1.0) + rhoL
        u[i] = (velR - velL)*0.5*( np.tanh(celcenter[i]/(1.0*dx)) + 1.0) + velL
        p[i] = (preR - preL)*0.5*( np.tanh(celcenter[i]/(1.0*dx)) + 1.0) + preL


    prepareanimation()

    addanimation(celcenter, rho, u, p,1)

    rhoL = np.zeros(ntot)   
    rhoR = np.zeros(ntot)  
    uL = np.zeros(ntot) 
    uR = np.zeros(ntot)   
    pL = np.zeros(ntot)  
    pR = np.zeros(ntot)
    
    pstar, ustar, rhostarL, rhostarR = starfun(rhoL, uL, pL, rhoR, uR, pR)

    astarL = np.sqrt(gamma*pstar/rhostarL)
    astarR = np.sqrt(gamma*pstar/rhostarR)

    time = 0.0
    dt = cfl*np.min(dx/(np.sqrt(gamma*p[ist:ien]/rho[ist:ien ]) + abs(u[ist:ien])))

    time_plot = 0.0
    dt_plot = 0.01


    while time < tout:

        for i in range(ist-1, ien):
            rhoL[i] = rho[i]
            uL[i] = u[i]
            pL[i] = p[i]

            rhoR[i] = rho[i+1]
            uR[i] = u[i+1]
            pR[i] = p[i+1]

    if s < ustar:

        if pL > pstar : # left fan

            sHL = uL - aL
            sTL = ustar - astarL

            if s < sHL:
                rho[i] = rhoL
                u[i] = uL
                p[i] = pL
            elif s > sTL:
                rho[i] = rhostarL
                u[i] = ustar
                p[i] = pstar
            else:
                rho[i] = rhoL*(2/gamma1 + gamma2/(gamma1*aL)*(uL-s))**(2/gamma2)
                u[i] = 2/gamma1*(aL + 0.5*gamma2*uL + s)
                p[i] = pL*(2/gamma1 + gamma2/(gamma1*aL)*(uL-s))**(2*gamma/gamma2)


        else: # left shock

            sL = uL - aL*np.sqrt(gamma1*pstar/pL + gamma4)

            if s < sL:
                rho[i] = rhoL
                u[i] = uL
                p[i] = pL
            else:
                rho[i] = rhostarL
                u[i] = ustar
                p[i] = pstar


    else: 
        if pR > pstar : # right fan

            sHR = uR + aR
            sTR = ustar + astarR

            if s > sHR:
                rho[i] = rhoR
                u[i] = uR
                p[i] = pR
            elif s < sTR:
                rho[i] = rhostarR
                u[i] = ustar
                p[i] = pstar
            else:
                rho[i] = rhoR*(2/gamma1 - gamma2/(gamma1*aR)*(uR-s))**(2/gamma2)
                u[i] = 2/gamma1*(-aR + 0.5*gamma2*uR + s)
                p[i] = pR*(2/gamma1 - gamma2/(gamma1*aR)*(uR-s))**(2*gamma/gamma2)

        else: # right shock
            sR = uR + aR*(np.sqrt(gamma5*pstar/pR + gamma4))

            if s > sR:
                rho[i] = rhoR
                u[i] = uR
                p[i] = pR
            else:
                rho[i] = rhostarR
                u[i] = ustar
                p[i] = pstar


    time = time + dt 

    time_plot = time_plot + dt
    if time_plot > dt_plot:
        addanimation(celcenter, rho, u, p,0)
        time_plot = 0.0

    dt = cfl*np.min(dx/(np.sqrt(gamma*p[ist:ien]/rho[ist:ien ]) + abs(u[ist:ien])))
    
    if time + dt > tout:
        dt = tout - time
    
    addanimation(celcenter, rho, u, p,0)


                


def starfun(rhoL, uL, pL, rhoR, uR, pR):
    global gamma, gamma1, gamma2, gamma3, gamma4, gamma5

    quser = 2
    pmax = max(pL, pR)
    pmin = min(pL, pR)
    aL = np.sqrt(gamma*pL/rhoL)
    aR = np.sqrt(gamma*pR/rhoR)
    q = pmax/pmin

    ###     PVRS      ####
    if q < quser:
        cL = rhoL * aL
        cR = rhoR * aR
        pstar = 1/(cL + cR) * (cL*pR + cR*pL + cL*cR*(uL - uR))
        pstar = max(pstar, 1e-6)

    ###     TRRS      ####
    if pstar < pmin:
        pstar = pow(((aL + aR - 0.5*gamma2*(uR - uL)) / (aL/pow(pL, gamma4) + aR/pow(pR, gamma4))),1/gamma4)
        

    ###     TSRS      ####
    if pstar > pmax:
        gL = np.sqrt(2/(gamma1*rhoL) * 1/(gamma3*pL + max((0.5*(pL + pR) + 0.5*(uR - uL)*0.25*(rhoL + rhoR)*(aL + aR), 1e-6))))
        gR = np.sqrt(2/(gamma1*rhoR) * 1/(gamma3*pR + max((0.5*(pL + pR) + 0.5*(uR - uL)*0.25*(rhoL + rhoR)*(aL + aR), 1e-6))))
        pstar = (gL*pL + gR*pR - (uR - uL)) / (gL + gR)



    if pstar > pL: #shock
        fL = (pstar - pL)*np.sqrt(2/(gamma1*rhoL) * 1/(gamma3*pL + pstar))
        rhostarL = rhoL * (pstar/pL + gamma2/gamma1) / ( (gamma2/gamma1)*pstar/pL + 1.0)
    else: #rarefaction
        fL =2*aL/gamma2 * (pow(pstar/pL, gamma4) - 1)
        rhostarL = rhoL * pow(pstar/pL, 1/gamma)
    if pstar > pR:
        fR = (pstar - pR)*np.sqrt(2/(gamma1*rhoR) * 1/(gamma3*pR + pstar))
        rhostarR = rhoR * (pstar/pR + gamma2/gamma1) / ( (gamma2/gamma1)*pstar/pR + 1.0)
    else:
        fR = 2*aR/gamma2 * (pow(pstar/pR, gamma4) - 1)
        rhostarR = rhoR * pow(pstar/pR, 1/gamma)

    ustar = 0.5*(uL + uR) + 0.5*(fR - fL)

    return pstar, ustar, rhostarL, rhostarR


exactsolution(rhoL = 1.0 ,velL= 0.0, preL=1.0 ,rhoR=0.125 ,velR=0.0 ,preR=0.1,L=1,ngrids=64,cfl=0.2,tout=0.2)

anim = ArtistAnimation(figA, artistA, interval=100)
plt.close(figA)
HTML(anim.to_jshtml())